## CREW AI

1. Open source framework for orchestrating multiple AI Agents 
2. These multiple agents collaborate like a team to complete the complex task
3. Each Agent is like a AI specialist  where they have dedicated roles and they collaborate automatically

Key facts:
- They are not dependent on Langchain or any other frameworks
- Optimized for speed hence it is production reliability
- There are two major core components: Crews(Automonous collaboration) and Flows(Structued and event driven control)
- Supports any LLM: Open AI, Anthropic Claude,Gemini, local models


## Difference of Crew AI vs other frameworks


- Crew AI is best to use when you have agents that have defined roles , High level API, Major difference is Fast, Standalone and lean
- Langgraph is best to use when you have Worflow that graph control and retries, Low level API, Major difference is StateUpdates and checkpoints
- Autogen is best to use when the agents based on the conversation patters and Medium level API, Major difference is Multi-agent Messaging

## https://docs.crewai.com/

## Installation

In [1]:
!pip3 install crewai crewai-tools

  Using cached mcp-1.26.0-py3-none-any.whl.metadata (89 kB)
  Using cached opentelemetry_api-1.34.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.34.1-py3-none-any.whl.metadata (1.6 kB)
  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.34.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_proto-1.34.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached opentelemetry_semantic_conventions-0.55b1-py3-none-any.whl.metadata (2.5 kB)
  Using cached pydantic_core-2.33.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.8 kB)
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of opentelemetry-exporter-otlp-proto-grpc to determine which version is compatible with other requirements. This could take a while.
  Using

## Setting up API Key

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable not set.")
else:
    print("OPENAI_API_KEY environment variable is set.")


OPENAI_API_KEY environment variable is set.


## Section 1 - Understanding Agents

An agent is an autonomous AI worker with a dedicated role , capability and purpose

An agent has fe key attributes:
- role ? - who is this agent? Ex. Senior Data analyst
- goal - what this agent trying to achieve
- backstory - The 'persona' that shapes the LLM behaviour
- tools - list of tools agent can use it
- llm - brain
- verbose - if True, prints each thought step by step
- allow_delegation - can hand off work to other agent
- memory - To rememeber across task
- max_iter - Maximum reasoning iterations before stopping

In [4]:
from crewai import Agent

researcher = Agent(name="Researcher", role="Senior Technology Researcher",
                    goal='Find accurate and up to date information about {topic} and identify the recent trends',
                    backstory = """ 
                     You are a senior technology researcher with expertise with 15 years of experience analyzing the industry trends
                     You are know for rigorous research and ability to find accurate and up to date information .
                     You always organise the finding in clear and conssise manner and note the confidence level of the information you find.
                      """,
                    verbose = True,
                    allow_delegation = False
                   )




In [5]:
writer = Agent(name="Writer", role="Senior Technical Writer",
               goal = 'Transform the research findings into a well-structured and engaging article that is easy to understand for a general audience',
               backstory = """
                You are a senior technical writer with expertise in translating complex technical information into clear and engaging content.
                You have a knack for storytelling and making technical topics accessible to a wide audience.
                You are known for your ability to create well-structured articles that are both informative and engaging.
                You write in simple english and avoid jargon to ensure that the content is easily understandable for a general audience.
                """,
                verbose = True,
                allow_delegation = False)

## TASK

A Task is a unit of work . Each task has

    - description - What should agent do?( Be specific)
    - expected_output - what result should look like?
    - agent - who is responsible?
    - context - if it has previous task does this depend on?

    

In [ ]:
from crewai import Task

research_task = Task(name = "Research Task",
                     description= """ Research the top 5 trends in {topic} for 2025.
                     
                     For Each trend, provide:
                     1. Trend Name (clear, concise, and descriptive)
                     2. What it is ( 2-3 lines of explnantion)
                     3. why it matters ( 2-3 lines of explanation)
                     4. one real life example (company, product or project or inittiatives)

                     Focus on trends that have clear eveidence on adoption , not just the speculations""",

                     expected_output=""" A sturcture report in markdown format with exactly 5 sections, one per trend. Each trend has header
                     then why/what/examples as subsection and Total length shoudl be 600-800 words""",

                     agent = researcher

                     )

In [8]:
writing_task = Task(name = "Writing Task",
                    description = """ Write an article based on the research report on {topic}

                    Your report should specify:
                    1. An Executive summary(3- 4 lines)
                    2.A brief introduction to the {topic} and its importance
                    3. 5 Key trends in {topic}
                    4. A conclusion with 2-3 actionable recommendations.
                    """,
                    expected_output = """ A well-structured article in markdown format with a clear flow and engaging writing style. 
                    The article should be between 800-1000 words. Must be ready to publish""",
                    agent = writer,
                    context=[research_task])

## TOOLS

- tools let agents to interact with the other knowledge beyond the LLM trained data.

- search web in real time
- read files
- write output to disk
- Execute code
- Query the database

Built in CrewAI Tools

- SerperDevTools ( Google Search)
- WebsiteSearcTool  ( Search/ Scrape a website)
- FileReadTool and FileWriteTool ( read and write output to local file)
- CodeInterpreterTool ( Run python code)

In [9]:
## Inbuilt Tools


from crewai_tools import FileWriterTool

file_writer = FileWriterTool()

file_writer_agent = Agent(name="File Writer", role="Senior Technical Writer",
                goal = 'Write the article to a markdown file and save them',
                backstory = """ Expert writer who produced high quality content and have experience with file handling and management and saves the output
                 to a disk for the client. """,
                 tools=[file_writer],
                 verbose = True,
                 allow_delegation = False)

In [17]:
## CUSTOM TOOLS

from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Optional, Type
import datetime

class DateToolInput(BaseModel):
    format: Optional[str] = Field(default="%Y-%m-%d", description="The format in which to return the current date. Default is '%Y-%m-%d'.")


class GetCurrentDateTool(BaseTool):
    name :str = "Get Current Date "
    description :str = "Returns the current date  in the specified format."
    args_schema: Type[BaseModel] = DateToolInput


    def _run(self, format: Optional[str] = "%Y-%m-%d") -> str:
        """Run the tool with the given input."""
        current_datetime = datetime.datetime.now().strftime(format)
        return current_datetime
    

date_tool = GetCurrentDateTool()


date_tool._run()


'2026-04-12'

## The Crew - Assembing the Team

The crew brings everything together:

- Which agents are on the team?
- what taks are need to be done?
- What process controls executables?


The crew brings agent and task together and defined how they work as the process

- agents
- tasks
- process ( Sequential and heirarical)
- verbose
- memory
- manager_llm
- planning , if true the crew creates the plan first and execute it




In [18]:
from crewai import Crew, Process

tech_report_crew = Crew(name="Tech Report Crew", agents=[researcher, writer], tasks=[research_task, writing_task], 
                        process = Process.sequential, verbose=True)

In [ ]:
tech_report_crew 

Crew(id=4c733aa6-e361-429a-8413-18fc8cec0b45, process=Process.sequential, number_of_agents=2, number_of_tasks=2)

In [27]:
print(f"Agents: {[agent.role for agent in tech_report_crew.agents]}")
print(f"Tasks: {[task.name for task in tech_report_crew.tasks]}")

Agents: ['Senior Technology Researcher', 'Senior Technical Writer']
Tasks: ['Research Task', 'Writing Task']


In [29]:
result = tech_report_crew.kickoff(inputs={"topic": "Artificial Intelligence in Healthcare"})

╭───────────────────────── 🚀 Crew Execution Started ──────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: Tech Report Crew                                                      │
│  ID: 4c733aa6-e361-429a-8413-18fc8cec0b45                                    │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────── 📋 Task Started ───────────────────────────────╮
│                                                                              │
│  Task Started                                                                │
│  Name: Research Task                                                         │
│  ID: 9ca2cb0b-e0de-4e57-a1b

In verbose:

Thought - when you agent does or needed any kind of reasoning though the problem.
Action - when your using tools
Action Input -  when your using tools
Observation -  when your using tools
Final Answer - The agent's final output for the task

## YAML CONFIGURATION

In production crewAI, you need to separate your agents/tasks definitions into YAML files.
This is the recommendation "golden path" by crew ai

Benifits:
- Bussiness team can read/edit agent defintions without touching the python code
- Easier for version control
- Modular way -  OOPS

In [ ]:
writer = Agent(name="Writer", role="Senior Technical Writer",
               goal = 'Transform the research findings into a well-structured and engaging article that is easy to understand for a general audience',
               backstory = """
                You are a senior technical writer with expertise in translating complex technical information into clear and engaging content.
                You have a knack for storytelling and making technical topics accessible to a wide audience.
                You are known for your ability to create well-structured articles that are both informative and engaging.
                You write in simple english and avoid jargon to ensure that the content is easily understandable for a general audience.
                """,
                verbose = True,
                allow_delegation = False)

In [32]:
import yaml
import os

os.makedirs("configs", exist_ok=True)

agent_yaml = """

researcher:
    role: Senior Technology Researcher
    goal: Find accurate and up to date information about {topic} and identify the recent trends
    backstory: >
    You are a senior technology researcher with expertise with 15 years of experience analyzing the industry trends
    You are know for rigorous research and ability to find accurate and up to date information .
    You always organise the finding in clear and conssise manner and note the confidence level of the information you find.
    

writer:
    role: Senior Technical Writer
    goal: Transform the research findings into a well-structured and engaging article that is easy to understand for a general audience
    backstory:  >
    You are a senior technical writer with expertise in translating complex technical information into clear and engaging content.
    You have a knack for storytelling and making technical topics accessible to a wide audience.
    You are known for your ability to create well-structured articles that are both informative and engaging.
    You write in simple english and avoid jargon to ensure that the content is easily understandable for a general audience.
"""


with open("configs/agents.yaml", "w") as f:
    f.write(agent_yaml)

In [33]:
task_yaml = """
research_task:
    description:  >
        Research the top 5 trends in {topic} for 2025.
                     
        For Each trend, provide:
        1. Trend Name (clear, concise, and descriptive)
        2. What it is ( 2-3 lines of explnantion)
        3. why it matters ( 2-3 lines of explanation)
        4. one real life example (company, product or project or inittiatives)   
        Focus on trends that have clear eveidence on adoption , not just the speculations
    
    expected_output: >
        A sturcture report in markdown format with exactly 5 sections, one per trend. Each trend has header
        then why/what/examples as subsection and Total length shoudl be 600-800 words


writing_task:
    description: >
        Write an article based on the research report on {topic}
        Your report should specify:
        1. An Executive summary(3- 4 lines)
        2. A brief introduction to the {topic} and its importance
        3. 5 Key trends in {topic}
        4. A conclusion with 2-3 actionable recommendations.

    expected_output: >
        A well-structured article in markdown format with a clear flow and engaging writing style. 
        The article should be between 800-1000 words. Must be ready to publish
"""

with open("configs/tasks.yaml", "w") as f:
    f.write(task_yaml)

In [ ]:
## Load from yaml and build crew

from crewai import Crew, Process, Agent, Task
import yaml

## Load YAML

with open("configs/agents.yaml", "r") as f:
    agents_config = yaml.safe_load(f)       

with open("configs/tasks.yaml", "r") as f:
    tasks_config = yaml.safe_load(f)


def build_agent(key):
    config = agents_config[key]
    return Agent(name=key.capitalize(), role=config["role"], goal=config["goal"], backstory=config["backstory"], verbose=True, allow_delegation=False)

def build_task(key):
    config = tasks_config[key]
    return Task(name=key.capitalize(), description=config["description"], expected_output=config["expected_output"], agent=build_agent("researcher") if "research" in key else build_agent("writer"))


researcher = build_agent("researcher")
writer = build_agent("writer")

research_task = build_task("research_task")
writing_task = build_task("writing_task")


##  Try this out!!!

1. News Summariser - Agent 1 research on the news topic , and agent 2 write a paragraph summary

2. Job application helper - Agent 1 analayses the job description and agent 2 will help to prepare a cover letter.



